# 🎬 AI Clipping Bot - Whop Campaign Automation
**Powered by Google Gemini 2.5 Flash + FFmpeg | 100% Gratuit**

---
### Comment utiliser ce notebook :
1. Remplissez vos clés API dans la **Cellule 1 (Config)**
2. Collez l'URL de votre brief Google Docs
3. Cliquez sur **Runtime → Run All** (ou Ctrl+F9)
4. Vos clips seront dans votre **Google Drive** !

> ⚠️ Gardez l'onglet Colab ouvert pendant le traitement.

In [ ]:
# ============================================================
# CELLULE 1 - CONFIGURATION (modifiez uniquement ici)
# ============================================================

GEMINI_API_KEY = 'COLLEZ_VOTRE_CLE_GEMINI_ICI'  # Votre cle Gemini

BRIEF_URL = 'https://docs.google.com/document/d/1modTZq5jJ5WkYZ1TmitNcV7MfHxTTRyGfKsFLdqCqUM/edit'  # URL du brief

CLIPS_TO_GENERATE = 20      # Nombre total de clips voulus
MIN_CLIP_DURATION  = 10     # Duree minimale en secondes
MAX_CLIP_DURATION  = 55     # Duree maximale en secondes

# Dossier Google Drive ou sauvegarder les clips (sera cree automatiquement)
DRIVE_OUTPUT_FOLDER = 'AI_Clips_Output'

print('Configuration OK')

In [ ]:
# ============================================================
# CELLULE 2 - INSTALLATION (a faire une seule fois)
# ============================================================
print('Installation des dependances...')
!apt-get install -y ffmpeg -q
!pip install -q google-generativeai google-api-python-client google-auth-httplib2 \
    google-auth-oauthlib gdown requests python-dotenv

import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print(f'FFmpeg: {result.stdout.split(chr(10))[0]}')
print('Installation complete !')

In [ ]:
# ============================================================
# CELLULE 3 - CONNEXION GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Creer le dossier de sortie dans Drive
OUTPUT_DIR = Path(f'/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR = Path('/content/downloads')
DOWNLOAD_DIR.mkdir(exist_ok=True)

print(f'Drive monte !')
print(f'Clips sauvegardes dans : {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELLULE 4 - LECTURE DU BRIEF GOOGLE DOCS
# ============================================================
import re
import requests
from html.parser import HTMLParser

def read_brief(doc_url):
    doc_id = re.search(r'/document/d/([a-zA-Z0-9_-]+)', doc_url).group(1)
    html = requests.get(
        f'https://docs.google.com/document/d/{doc_id}/export?format=html',
        timeout=30
    ).text

    class Parser(HTMLParser):
        def __init__(self):
            super().__init__()
            self.text = []
            self.links = []
        def handle_starttag(self, tag, attrs):
            if tag == 'a':
                href = dict(attrs).get('href', '')
                if 'google.com/url?q=' in href:
                    from urllib.parse import unquote
                    m = re.search(r'q=([^&]+)', href)
                    if m: href = unquote(m.group(1))
                if href: self.links.append(href)
        def handle_data(self, d):
            self.text.append(d)

    p = Parser()
    p.feed(html)
    text = ' '.join(p.text)

    drive_links = []
    for href in p.links:
        if 'drive.google.com' in href and href not in drive_links:
            drive_links.append(href.rstrip('.,;)'))

    raw = re.findall(r'https://drive\.google\.com/[^\s"\'\'<>&]+', html)
    for l in raw:
        c = l.rstrip('.,;)')
        if c not in drive_links:
            drive_links.append(c)

    print(f'Brief lu : {len(drive_links)} lien(s) Drive trouves')
    for l in drive_links:
        print(f'  - {l[:80]}')
    return {'text': text, 'drive_links': drive_links}

brief = read_brief(BRIEF_URL)
print(f'\nApercu: {brief["text"][:300]}')

In [ ]:
# ============================================================
# CELLULE 5 - TELECHARGEMENT DES VIDEOS DEPUIS DRIVE
# ============================================================
import time, json
import gdown

def get_folder_id(url):
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', url)
    return m.group(1) if m else None

def get_file_id(url):
    for pat in [r'/file/d/([a-zA-Z0-9_-]+)', r'[?&]id=([a-zA-Z0-9_-]+)']:
        m = re.search(pat, url)
        if m: return m.group(1)
    return None

def download_robust(file_id, output_path, max_retries=5):
    already = output_path.stat().st_size if output_path.exists() else 0
    if already > 500_000:
        print(f'  [SKIP] {output_path.name} deja telecharge ({already/(1024*1024):.1f} MB)')
        return True
    for attempt in range(1, max_retries+1):
        try:
            url = f'https://drive.google.com/uc?export=download&id={file_id}&confirm=t'
            headers = {'Range': f'bytes={already}-'} if already > 0 else {}
            r = requests.get(url, headers=headers, stream=True, timeout=60)
            total = int(r.headers.get('content-length', 0)) + already
            with open(output_path, 'ab' if already > 0 else 'wb') as f:
                written = 0
                for chunk in r.iter_content(1024*512):
                    if chunk:
                        f.write(chunk)
                        written += len(chunk)
                        if total > 0:
                            pct = (already+written)*100//total
                            print(f'\r  {pct}% {(already+written)/(1024*1024):.1f}/{total/(1024*1024):.1f} MB  [{output_path.name[:40]}]', end='', flush=True)
            print()
            if output_path.exists() and output_path.stat().st_size > 500_000:
                return True
        except Exception as e:
            print(f'\n  Retry {attempt}: {e}')
            already = output_path.stat().st_size if output_path.exists() else 0
            time.sleep(3)
    return False

VIDEO_EXT = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
seen_folders = set()

for url in brief['drive_links']:
    folder_id = get_folder_id(url)
    file_id   = get_file_id(url)

    if folder_id and folder_id not in seen_folders:
        seen_folders.add(folder_id)
        print(f'\nDossier : {folder_id}')
        try:
            # Lister le dossier avec gdown
            files = gdown.download_folder(
                url=f'https://drive.google.com/drive/folders/{folder_id}',
                output=str(DOWNLOAD_DIR) + '/',
                quiet=True,
                use_cookies=False,
                remaining_ok=True
            ) or []
            print(f'  gdown: {len(files)} fichier(s)')
        except Exception as e:
            print(f'  Erreur gdown folder: {e}')
    elif file_id and not folder_id:
        out = DOWNLOAD_DIR / f'{file_id}.mp4'
        download_robust(file_id, out)

video_files = sorted([
    p for p in DOWNLOAD_DIR.rglob('*')
    if p.suffix.lower() in VIDEO_EXT and p.stat().st_size > 500_000
])
print(f'\n{len(video_files)} video(s) disponibles pour analyse :')
for v in video_files:
    print(f'  {v.name} ({v.stat().st_size/(1024*1024):.1f} MB)')

In [ ]:
# ============================================================
# CELLULE 6 - ANALYSE IA AVEC GEMINI 2.5 FLASH
# ============================================================
import google.generativeai as genai
import subprocess, json, time

genai.configure(api_key=GEMINI_API_KEY)

SYSTEM_PROMPT = """
Tu es un expert en montage video viral pour TikTok et Instagram Reels.
Analyse la video et selectionne les meilleurs segments selon le brief.
REGLE : reponds UNIQUEMENT en JSON valide, sans texte autour.
FORMAT:
{"clips": [{"id": 1, "start": 5.0, "end": 30.0, "duration": 25.0, "score": 90, "reason": "Moment fort"}],
 "video_summary": "Description courte"}
"""

def get_duration(path):
    r = subprocess.run(['ffprobe', '-v', 'quiet', '-print_format', 'json',
                        '-show_format', str(path)], capture_output=True, text=True)
    try: return float(json.loads(r.stdout)['format']['duration'])
    except: return 60.0

def analyze_video(video_path, n_clips, min_dur, max_dur):
    print(f'  Upload vers Gemini...')
    vf = genai.upload_file(path=str(video_path), display_name=video_path.name)
    while vf.state.name == 'PROCESSING':
        time.sleep(3)
        vf = genai.get_file(vf.name)
    if vf.state.name == 'FAILED':
        raise RuntimeError('Gemini ne peut pas traiter cette video')

    model = genai.GenerativeModel('gemini-2.5-flash', system_instruction=SYSTEM_PROMPT)
    prompt = f"""
Brief de campagne:
{brief['text'][:3000]}

MISSION: Selectionne {n_clips} segments de cette video conformes au brief.
Duree: {min_dur}s a {max_dur}s par clip. Format: vertical 9:16 TikTok.
Reponds en JSON pur uniquement.
"""
    resp = model.generate_content([vf, prompt],
        generation_config=genai.GenerationConfig(temperature=0.2, max_output_tokens=2048))

    try:
        genai.delete_file(vf.name)
    except: pass

    raw = resp.text.strip()
    raw = re.sub(r'^```(?:json)?\n?', '', raw)
    raw = re.sub(r'\n?```$', '', raw)
    return json.loads(raw).get('clips', [])

# Calculer durees et repartir les clips proportionnellement
durations = {str(v): get_duration(v) for v in video_files}
total_dur = sum(durations.values())

print(f'Videos a analyser: {len(video_files)}')
for v in video_files:
    print(f'  {v.name}: {durations[str(v)]:.1f}s')

analysis_results = {}
for i, video_path in enumerate(video_files):
    dur = durations[str(video_path)]
    n = max(1, round((dur / total_dur) * CLIPS_TO_GENERATE))
    n = min(n, max(1, int(dur // MIN_CLIP_DURATION)))

    print(f'\n[{i+1}/{len(video_files)}] {video_path.name} -> {n} clips')
    try:
        clips = analyze_video(video_path, n, MIN_CLIP_DURATION, MAX_CLIP_DURATION)
        analysis_results[str(video_path)] = clips
        print(f'  OK: {len(clips)} clips trouves')
    except Exception as e:
        print(f'  ERREUR: {e}')
        analysis_results[str(video_path)] = []

total = sum(len(c) for c in analysis_results.values())
print(f'\nTotal: {total} clips identifies par Gemini')

In [ ]:
# ============================================================
# CELLULE 7 - MONTAGE VIDEO AVEC FFMPEG (9:16 vertical)
# ============================================================
from datetime import datetime

campaign_name = 'soul_tied'
created_clips = []

for video_path_str, clips in analysis_results.items():
    if not clips: continue
    video_path = Path(video_path_str)
    dur_total = durations[video_path_str]

    for clip in clips:
        start = float(clip.get('start', 0))
        end   = min(float(clip.get('end', start+30)), dur_total)
        cid   = clip.get('id', len(created_clips)+1)

        ts = datetime.now().strftime('%H%M%S')
        out_name = f'{campaign_name}_clip_{cid:02d}_{ts}.mp4'
        out_path = OUTPUT_DIR / out_name

        # Filtre FFmpeg : scale hauteur 1920, crop 1080 centre
        vf = 'scale=-2:1920,crop=1080:1920'
        cmd = [
            'ffmpeg', '-y',
            '-ss', str(start),
            '-i', str(video_path),
            '-t', str(end - start),
            '-vf', vf,
            '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
            '-c:a', 'aac', '-b:a', '128k',
            '-movflags', '+faststart',
            str(out_path)
        ]
        result = subprocess.run(cmd, capture_output=True, timeout=300)
        if result.returncode == 0 and out_path.exists():
            size_mb = out_path.stat().st_size / (1024*1024)
            print(f'  OK: {out_name} ({size_mb:.1f} MB)')
            created_clips.append(out_path)
        else:
            print(f'  ERREUR FFmpeg sur clip {cid}: {result.stderr[-200:]}')

print(f'\n{len(created_clips)} clips crees dans Google Drive: {OUTPUT_DIR}')

In [ ]:
# ============================================================
# CELLULE 8 - RAPPORT FINAL
# ============================================================
print('=' * 55)
print('  RAPPORT FINAL - AI Clipping Bot')
print('=' * 55)
print(f'  Videos analysees  : {len(video_files)}')
print(f'  Clips generes     : {len(created_clips)}')
print(f'  Dossier de sortie : Google Drive / {DRIVE_OUTPUT_FOLDER}')
print('=' * 55)
print('\nFichiers crees :')
for p in created_clips:
    mb = p.stat().st_size / (1024*1024)
    print(f'  {p.name} ({mb:.1f} MB)')
print('\nVos clips sont dans Google Drive, prets a poster !')